In [2]:
import yaml
import os
import torch
import gc
from ultralytics import YOLO

# 강제 메모리 해제 함수 정의
def force_clear_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()
    print("🧹 [Memory Cleanup] 현재 셀의 가비지 및 GPU 캐시를 완벽히 정리했습니다.")

# 에폭마다 실행될 콜백 함수
def on_epoch_end_clean(trainer):
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

In [2]:
import os
import gc
import torch
import pandas as pd
import glob
from ultralytics import YOLO

def force_clear_memory():
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

def find_best_pt(search_dir):
    """지정된 디렉토리 하위에서 best.pt 파일을 찾아 경로를 반환합니다."""
    search_path = os.path.join(search_dir, '**', 'best.pt')
    files = glob.glob(search_path, recursive=True)
    if files:
        # 여러 개가 있을 경우 가장 최근에 생성된(수정된) 파일 선택
        return max(files, key=os.path.getmtime)
    return None

def run_pipeline():
    target_models = {
        'nano': 'yolov8n.pt',
        'small': 'yolov8s.pt',
        'medium': 'yolov8m.pt'
    }
    
    results = []
    DATA_PATH = 'data.yaml'

    for label, model_name in target_models.items():
        print(f"\n{'='*20} [시작] {label.upper()} 모델 파이프라인 {'='*20}")
        
        # 1. 튜닝 시작
        print(f"🚀 {label.upper()} 튜닝 시작...")
        model = YOLO(model_name)
        run_name = f'tune_{label}'
        
        model.tune(
            data=DATA_PATH, epochs=10, iterations=1, imgsz=640,
            device='mps', batch=8, workers=2, amp=False, plots=False, save=True,
            project='runs/tune_cells', name=run_name
        )
        
        # 2. 동적 경로 찾기
        best_weight_path = find_best_pt(os.path.join('runs/tune_cells', run_name))
        
        if not best_weight_path:
            print(f"❌ {label.upper()} 모델의 best.pt를 찾을 수 없습니다. 튜닝 실패 가능성 확인 필요.")
            continue
            
        print(f"✅ 모델 발견: {best_weight_path}")
        
        # 3. 테스트셋 평가
        print(f"⏳ {label.upper()} 테스트셋 평가 중...")
        best_model = YOLO(best_weight_path)
        metrics = best_model.val(data=DATA_PATH, split='test', imgsz=640, device='mps', verbose=False)
        
        results.append({
            "Model": label.upper(),
            "mAP50": metrics.results_dict['metrics/mAP50(B)'],
            "mAP50-95": metrics.results_dict['metrics/mAP50-95(B)'],
            "Precision": metrics.results_dict['metrics/precision(B)'],
            "Recall": metrics.results_dict['metrics/recall(B)']
        })
        
        del model
        del best_model
        force_clear_memory()

    # 4. 최종 결과 출력
    print("\n" + "="*80)
    print("🏆 [최종 성적표] 3개 모델 테스트셋 평가 결과")
    print("="*80)
    df = pd.DataFrame(results).sort_values(by="mAP50", ascending=False)
    print(df.to_markdown(index=False, numalign="left", stralign="left"))
    print("="*80)

if __name__ == '__main__':
    run_pipeline()


==================== [시작] NANO 모델 파이프라인 ====================
🚀 NANO 튜닝 시작...
Tuner: Initialized Tuner instance with 'tune_dir=/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_nano-2'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/1 with hyperparameters: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'box': 7.5, 'cls': 0.5, 'cls_pw': 0.0, 'dfl': 1.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'bgr': 0.0, 'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 10}
Ultralytics 8.4.65 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M5)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10

KeyError: 'mAP50'

In [3]:
import os
import gc
import torch
import pandas as pd
import glob
from ultralytics import YOLO

# 1. 환경 설정
BASE_DIR = "/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells"
DATA_YAML = "data.yaml"

# 타겟 모델 폴더명 (본인의 실제 폴더명과 다를 경우 수정하세요)
TARGET_MODELS = {
    "Nano": "tune_nano",
    "Small": "tune_small",
    "Medium": "tune_medium"
}

results_list = []

print("🏁 [평가 모드] 나노/스몰/미듐 모델 테스트셋 채점 시작...\n")

for model_name, folder_name in TARGET_MODELS.items():
    # 폴더 내에서 best.pt 자동 탐색 (가장 최근에 수정된 파일)
    search_path = os.path.join(BASE_DIR, folder_name, "**", "best.pt")
    files = glob.glob(search_path, recursive=True)
    
    if not files:
        print(f"⚠️ [{model_name}] 해당 폴더에 best.pt가 없습니다: {os.path.join(BASE_DIR, folder_name)}")
        continue
    
    # 최신 파일 선택
    best_pt_path = max(files, key=os.path.getmtime)
    print(f"🔍 [{model_name}] 모델 로드: {best_pt_path}")
    
    try:
        # 모델 객체 생성 및 평가
        model = YOLO(best_pt_path)
        metrics = model.val(
            data=DATA_YAML,
            split='test',    # 훈련 데이터가 아닌 테스트셋으로 평가
            imgsz=640,       # 640 고정 평가
            device='mps',    # Mac M5 사용
            verbose=False
        )
        
        # 성능 지표 수집
        results_list.append({
            "모델": model_name,
            "mAP50": round(metrics.box.map50, 4),
            "mAP50-95": round(metrics.box.map, 4),
            "Precision": round(metrics.box.mp, 4),
            "Recall": round(metrics.box.mr, 4)
        })
        
        # 메모리 정리
        del model
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
            
    except Exception as e:
        print(f"❌ [{model_name}] 평가 중 오류 발생: {e}")

# 2. 결과 출력
print("\n" + "="*70)
print("🏆 [최종 성적표] Nano, Small, Medium 테스트셋 평가 결과")
print("="*70)

if results_list:
    df = pd.DataFrame(results_list)
    df = df.sort_values(by="mAP50", ascending=False).reset_index(drop=True)
    df.index = df.index + 1
    df.index.name = "순위"
    
    print(df.to_markdown())
    print("="*70)
    print(f"🎯 1등 모델: {df.iloc[0]['모델']} (mAP50: {df.iloc[0]['mAP50']})")
else:
    print("❌ 평가 결과가 없습니다. 폴더 경로와 best.pt 존재 여부를 확인하세요.")

🏁 [평가 모드] 나노/스몰/미듐 모델 테스트셋 채점 시작...

🔍 [Nano] 모델 로드: /Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_nano/weights/best.pt
Ultralytics 8.4.65 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M5)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.1±0.1 ms, read: 227.8±86.2 MB/s, size: 42.2 KB)
val: Scanning /Users/gyuminkang/Desktop/YOLO/test/labels.cache... 458 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 458/458 80.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 2.7it/s 10.5s0.3s
                   all        458       6222      0.763        0.7      0.756      0.441
Speed: 0.2ms preprocess, 2.4ms inference, 0.0ms loss, 6.6ms postprocess per image
Results saved to /Users/gyuminkang/Desktop/YOLO/runs/detect/val-42
🔍 [Small] 모델 로드: /Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_small/weights/best.pt
Ultralytics 8.4.65

In [4]:
import os
import gc
import torch
import pandas as pd
import glob
from ultralytics import YOLO

# 1. 환경 설정
BASE_DIR = "/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells"
DATA_YAML = "data.yaml"

# 타겟 모델 폴더명 (실제 폴더명과 일치하는지 확인하세요)
TARGET_MODELS = {
    "Nano": "tune_nano",
    "Small": "tune_small",
    "Medium": "tune_medium"
}

results_list = []

print("🏁 [평가 모드] 나노/스몰/미듐 모델 테스트셋 채점 시작...\n")

for model_name, folder_name in TARGET_MODELS.items():
    # 폴더 내에서 best.pt 자동 탐색
    search_path = os.path.join(BASE_DIR, folder_name, "**", "best.pt")
    files = glob.glob(search_path, recursive=True)
    
    if not files:
        print(f"⚠️ [{model_name}] 해당 폴더에 best.pt가 없습니다: {os.path.join(BASE_DIR, folder_name)}")
        continue
    
    best_pt_path = max(files, key=os.path.getmtime)
    print(f"🔍 [{model_name}] 모델 로드: {best_pt_path}")
    
    try:
        # 모델 객체 생성 및 평가
        model = YOLO(best_pt_path)
        metrics = model.val(
            data=DATA_YAML,
            split='test',    # 테스트셋 평가
            imgsz=640,       # 640 고정
            device='mps',    # Mac GPU 사용
            verbose=False
        )
        
        # 성능 및 추론 속도 지표 수집
        results_list.append({
            "모델": model_name,
            "mAP50": round(metrics.box.map50, 4),
            "mAP50-95": round(metrics.box.map, 4),
            "Precision": round(metrics.box.mp, 4),
            "Recall": round(metrics.box.mr, 4),
            "추론속도(ms)": round(metrics.speed['inference'], 2)
        })
        
        # 메모리 정리
        del model
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()
            
    except Exception as e:
        print(f"❌ [{model_name}] 평가 중 오류 발생: {e}")

# 2. 결과 출력
print("\n" + "="*85)
print("🏆 [최종 성적표] 모델별 성능 및 추론 속도 비교")
print("="*85)

if results_list:
    df = pd.DataFrame(results_list)
    df = df.sort_values(by="mAP50", ascending=False).reset_index(drop=True)
    df.index = df.index + 1
    df.index.name = "순위"
    
    print(df.to_markdown())
    print("="*85)
    print(f"🎯 가장 성능이 좋은 모델: {df.iloc[0]['모델']} (mAP50: {df.iloc[0]['mAP50']}, 속도: {df.iloc[0]['추론속도(ms)']}ms)")
else:
    print("❌ 평가 결과가 없습니다. 폴더 경로와 best.pt 존재 여부를 확인하세요.")

🏁 [평가 모드] 나노/스몰/미듐 모델 테스트셋 채점 시작...

🔍 [Nano] 모델 로드: /Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_nano/weights/best.pt
Ultralytics 8.4.65 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M5)
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2143.4±213.0 MB/s, size: 48.3 KB)
val: Scanning /Users/gyuminkang/Desktop/YOLO/test/labels.cache... 458 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 458/458 480.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 6.0it/s 4.8s0.2s
                   all        458       6222      0.763        0.7      0.756      0.441
Speed: 0.2ms preprocess, 2.4ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /Users/gyuminkang/Desktop/YOLO/runs/detect/val-45
🔍 [Small] 모델 로드: /Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_small/weights/best.pt
Ultralytics 8.4.

In [5]:
from ultralytics import YOLO
import os

def export_to_onnx_custom(source_weights, new_filename):
    """
    source_weights: 원본 best.pt 파일 경로
    new_filename: 저장하고 싶은 이름 (예: 'my_medium_model.onnx')
    """
    
    # 1. 파일 존재 여부 확인
    if not os.path.exists(source_weights):
        print(f"❌ 가중치 파일을 찾을 수 없습니다: {source_weights}")
        return

    print(f"📦 모델 변환 시작: {source_weights}")
    model = YOLO(source_weights)
    
    # 2. ONNX로 export (기본값은 best.onnx로 생성됨)
    exported_path = model.export(format='onnx', imgsz=640, simplify=True)
    
    # 3. 파일 이름 변경 및 경로 재지정
    directory = os.path.dirname(exported_path)
    new_path = os.path.join(directory, new_filename)
    
    # 기존 파일이 있다면 삭제 (덮어쓰기 방지)
    if os.path.exists(new_path):
        os.remove(new_path)
        
    os.rename(exported_path, new_path)
    
    # 4. 결과 출력 (절대 경로)
    full_path = os.path.abspath(new_path)
    print("\n" + "="*50)
    print("✅ 변환 성공!")
    print(f"파일명: {new_filename}")
    print(f"저장 경로: {full_path}")
    print("="*50)

if __name__ == "__main__":
    # Medium 모델 경로 (사용자님 환경에 맞게 수정 가능)
    SOURCE = '/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.pt'
    # 원하는 파일 이름
    NEW_NAME = 'car_detect_medium_v1.onnx'
    
    export_to_onnx_custom(SOURCE, NEW_NAME)

📦 모델 변환 시작: /Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.pt
Ultralytics 8.4.65 🚀 Python-3.11.15 torch-2.12.0 CPU (Apple M5)
Model summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from '/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (49.6 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/18.0 MB ? eta -:--:--
   ━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/18.0 MB 26.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━ 10.5/18.0 MB 27.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━ 13.4/18.0 MB 22.3 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 23.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [6]:
from ultralytics import YOLO
import pandas as pd
import torch
import gc

def benchmark_models(pt_path, onnx_path, data_yaml):
    results = []
    
    # 평가할 모델 리스트
    models_to_test = {
        "PyTorch (Original)": pt_path,
        "ONNX (Converted)": onnx_path
    }

    print(f"🏁 [벤치마크 시작] 모델별 성능 및 속도 평가 중...\n")

    for name, path in models_to_test.items():
        print(f"⏳ [{name}] 평가 중...")
        
        # 모델 로드
        model = YOLO(path)
        
        # 테스트셋 평가 (imgsz=640 고정)
        metrics = model.val(
            data=data_yaml,
            split='test',
            imgsz=640,
            device='mps', # Mac 환경
            verbose=False
        )
        
        # 데이터 수집
        results.append({
            "모델 포맷": name,
            "mAP50": round(metrics.box.map50, 4),
            "mAP50-95": round(metrics.box.map, 4),
            "Precision": round(metrics.box.mp, 4),
            "Recall": round(metrics.box.mr, 4),
            "추론속도(ms)": round(metrics.speed['inference'], 2)
        })
        
        # 메모리 정리
        del model
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

    # 결과 데이터프레임 생성 및 출력
    df = pd.DataFrame(results)
    print("\n" + "="*80)
    print("🏆 [벤치마크 결과] 원본 vs 변환 모델 비교")
    print("="*80)
    print(df.to_markdown(index=False))
    print("="*80)

if __name__ == "__main__":
    # 각 경로를 정확히 입력하세요
    PT_WEIGHTS = '/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.pt'
    ONNX_WEIGHTS = '/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/car_detect_medium_v1.onnx' # 아까 저장한 파일명
    DATA_YAML = 'data.yaml'
    
    benchmark_models(PT_WEIGHTS, ONNX_WEIGHTS, DATA_YAML)

🏁 [벤치마크 시작] 모델별 성능 및 속도 평가 중...

⏳ [PyTorch (Original)] 평가 중...
Ultralytics 8.4.65 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M5)
Model summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2441.3±724.7 MB/s, size: 43.0 KB)
val: Scanning /Users/gyuminkang/Desktop/YOLO/test/labels.cache... 458 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 458/458 213.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 29/29 2.4it/s 12.1s0.4s
                   all        458       6222      0.761      0.783      0.784      0.466
Speed: 0.1ms preprocess, 12.9ms inference, 0.0ms loss, 7.9ms postprocess per image
Results saved to /Users/gyuminkang/Desktop/YOLO/runs/detect/val-48
⏳ [ONNX (Converted)] 평가 중...
Ultralytics 8.4.65 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M5)
Loading /Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/car_detec

In [7]:
from ultralytics import YOLO
import pandas as pd
import torch
import gc
import os

def benchmark_all_formats(pt_path, data_yaml):
    # 1. 경로 설정 및 파일 이름 정의
    dir_path = os.path.dirname(pt_path)
    onnx_path = os.path.join(dir_path, "model_onnx.onnx")
    coreml_path = os.path.join(dir_path, "model_coreml.mlpackage")
    
    # 2. 파일이 없으면 자동으로 변환 (이미 있으면 건너뜀)
    model_orig = YOLO(pt_path)
    
    if not os.path.exists(onnx_path):
        print(f"📦 ONNX 변환 시작...")
        model_orig.export(format='onnx', imgsz=640, simplify=True)
        # Ultralytics는 export 시 원본 경로에 저장함, 필요시 이름 변경 로직 추가 가능
        # 여기서는 편의상 생성된 경로를 찾아서 지정
        onnx_path = os.path.join(dir_path, "best.onnx") 
        
    if not os.path.exists(coreml_path):
        print(f"📦 CoreML 변환 시작...")
        model_orig.export(format='coreml', imgsz=640)
        coreml_path = os.path.join(dir_path, "best.mlpackage")

    # 3. 평가할 모델 리스트
    targets = {
        "PyTorch": pt_path,
        "ONNX": onnx_path,
        "CoreML": coreml_path
    }
    
    results = []
    
    print(f"\n🏁 [벤치마크 시작] PyTorch vs ONNX vs CoreML\n")
    
    for name, path in targets.items():
        print(f"⏳ [{name}] 평가 중...")
        
        # 모델 로드 (Ultralytics가 자동으로 포맷 인식)
        model = YOLO(path)
        
        # 테스트셋 평가
        metrics = model.val(
            data=data_yaml,
            split='test',
            imgsz=640,
            device='mps', 
            verbose=False
        )
        
        results.append({
            "포맷": name,
            "mAP50": round(metrics.box.map50, 4),
            "mAP50-95": round(metrics.box.map, 4),
            "Precision": round(metrics.box.mp, 4),
            "Recall": round(metrics.box.mr, 4),
            "추론속도(ms)": round(metrics.speed['inference'], 2)
        })
        
        del model
        gc.collect()
        if torch.backends.mps.is_available():
            torch.mps.empty_cache()

    # 4. 결과 출력
    df = pd.DataFrame(results)
    print("\n" + "="*85)
    print("🏆 [최종 성적표] 모델 포맷별 성능 비교")
    print("="*85)
    print(df.to_markdown(index=False))
    print("="*85)

if __name__ == "__main__":
    # 본인의 Medium 모델 경로를 정확히 입력하세요
    PT_FILE = '/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.pt'
    benchmark_all_formats(PT_FILE, 'data.yaml')

📦 ONNX 변환 시작...
Ultralytics 8.4.65 🚀 Python-3.11.15 torch-2.12.0 CPU (Apple M5)
Model summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs

PyTorch: starting from '/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (49.6 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 0.7s, saved as '/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.onnx' (98.8 MB)

Export complete (1.0s)
Results saved to /Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.onnx
Predict:         yolo predict task=detect model=/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.onnx im

Running MIL default pipeline:   0%|          | 0/95 [00:00<?, ? passes/s]/opt/miniconda3/envs/ai_new/lib/python3.11/site-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '1136', of the source model, has been renamed to 'var_1136' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 332.10 passes/s]


CoreML: export success ✅ 9.8s, saved as '/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.mlpackage' (49.5 MB)

Export complete (10.1s)
Results saved to /Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.mlpackage
Predict:         yolo predict task=detect model=/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.mlpackage imgsz=640 
Validate:        yolo val task=detect model=/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.mlpackage imgsz=640 data=data.yaml  
Visualize:       https://netron.app

🏁 [벤치마크 시작] PyTorch vs ONNX vs CoreML

⏳ [PyTorch] 평가 중...
Ultralytics 8.4.65 🚀 Python-3.11.15 torch-2.12.0 MPS (Apple M5)
Model summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2375.8±607.3 MB/s, size: 43.3 KB)
val: Scanning /Users/gyuminkang/Desktop/YOLO/test/labels.cache... 458 imag

In [ ]:
import cv2
import os
from ultralytics import YOLO

# 1. 경로 설정
MODEL_PATH = '/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.mlpackage'
INPUT_VIDEO = '/Users/gyuminkang/Desktop/YOLO/cctv.mp4'
OUTPUT_VIDEO = '/Users/gyuminkang/Desktop/YOLO/output_final.mp4'

# 클래스 고정 및 색상 저장소
class_memory = {}
class_colors = {}

def get_color(cls_id):
    """클래스 ID별 고정 색상 생성 (반드시 int로 변환)"""
    if cls_id not in class_colors:
        # 계산 결과가 numpy형일 수 있으므로 int()로 확실하게 감싸줍니다.
        b = int((cls_id * 100) % 255)
        g = int((cls_id * 200) % 255)
        r = int((cls_id * 50) % 255)
        class_colors[cls_id] = (b, g, r)
    return class_colors[cls_id]

def draw_custom_box(frame, box, track_id, cls_id, conf, class_names):
    """클래스 고정 및 스타일 적용"""
    # 1. 클래스 고정 로직
    if track_id not in class_memory:
        class_memory[track_id] = cls_id
    else:
        stored_cls = class_memory[track_id]
        stored_name = class_names[stored_cls].lower()
        current_name = class_names[cls_id].lower()
        
        # '차'로 바뀌는 것 방어
        if ('bus' in stored_name or 'truck' in stored_name) and 'car' in current_name:
            cls_id = stored_cls
        elif 'bus' in current_name or 'truck' in current_name:
            class_memory[track_id] = cls_id
    
    # 2. 색상 및 좌표 변환 (int() 강제 적용)
    color = get_color(cls_id)
    x1, y1, x2, y2 = map(int, box)
    label = f"{class_names[cls_id].upper()} {conf:.2f}"
    
    # 3. 그리기
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    
    # 반투명 라벨 배경 (int() 확인)
    (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    overlay = frame.copy()
    cv2.rectangle(overlay, (x1, y1 - 20), (x1 + text_w, y1), color, -1)
    cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
    
    cv2.putText(frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

def run_final_demo():
    model = YOLO(MODEL_PATH)
    cap = cv2.VideoCapture(INPUT_VIDEO)
    
    # 영상 설정
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    
    print("🎬 시연 시작! 종료는 'q'를 누르세요.")

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            results = model.track(frame, device='mps', persist=True, verbose=False)
            
            if results[0].boxes.id is not None:
                # 데이터를 명확하게 일반 정수 리스트/배열로 추출
                boxes = results[0].boxes.xyxy.cpu().numpy()
                cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
                track_ids = results[0].boxes.id.cpu().numpy().astype(int)
                confs = results[0].boxes.conf.cpu().numpy()
                
                for box, track_id, cls_id, conf in zip(boxes, track_ids, cls_ids, confs):
                    draw_custom_box(frame, box, track_id, cls_id, conf, model.names)
            
            cv2.imshow("Final CCTV Demo", frame)
            out.write(frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        out.release()
        cv2.destroyAllWindows()
        print(f"✅ 저장 완료! 파일 확인: {OUTPUT_VIDEO}")

if __name__ == "__main__":
    run_final_demo()

🎬 시연 시작! 종료는 'q'를 누르세요.
Loading /Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.mlpackage for CoreML inference...
✅ 저장 완료! 파일 확인: /Users/gyuminkang/Desktop/YOLO/output_final.mp4


In [21]:
import cv2
import os
from ultralytics import YOLO

# 1. 경로 설정
MODEL_PATH = '/Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.mlpackage'
INPUT_VIDEO = '/Users/gyuminkang/Desktop/YOLO/cctv.mp4'
OUTPUT_VIDEO = '/Users/gyuminkang/Desktop/YOLO/output_final.mp4'

# 클래스 고정 및 색상 저장소
class_memory = {}
class_colors = {}

def get_color(cls_id):
    """클래스 ID별 파란색, 초록색, 빨간색 고정 색상 생성"""
    if cls_id not in class_colors:
        # OpenCV는 BGR 형식을 사용합니다: 파랑(255,0,0), 초록(0,255,0), 빨강(0,0,255)
        palette = [(255, 0, 0), (0, 255, 0), (0, 0, 255)]
        # 클래스 ID를 3으로 나눈 나머지를 사용하여 세 가지 색상을 순환 할당
        class_colors[cls_id] = palette[int(cls_id) % 3]
    return class_colors[cls_id]

def draw_custom_box(frame, box, track_id, cls_id, conf, class_names):
    """클래스 고정 및 스타일 적용"""
    # 1. 클래스 고정 로직
    if track_id not in class_memory:
        class_memory[track_id] = cls_id
    else:
        stored_cls = class_memory[track_id]
        stored_name = class_names[stored_cls].lower()
        current_name = class_names[cls_id].lower()
        
        # '차'로 바뀌는 것 방어
        if ('bus' in stored_name or 'truck' in stored_name) and 'car' in current_name:
            cls_id = stored_cls
        elif 'bus' in current_name or 'truck' in current_name:
            class_memory[track_id] = cls_id
    
    # 2. 색상 및 좌표 변환 (int() 강제 적용)
    color = get_color(cls_id)
    x1, y1, x2, y2 = map(int, box)
    label = f"{class_names[cls_id].upper()} {conf:.2f}"
    
    # 3. 그리기
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
    
    # 반투명 라벨 배경 (int() 확인)
    (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    overlay = frame.copy()
    cv2.rectangle(overlay, (x1, y1 - 20), (x1 + text_w, y1), color, -1)
    cv2.addWeighted(overlay, 0.4, frame, 0.6, 0, frame)
    
    cv2.putText(frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

def run_final_demo():
    model = YOLO(MODEL_PATH)
    cap = cv2.VideoCapture(INPUT_VIDEO)
    
    # 영상 설정
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    out = cv2.VideoWriter(OUTPUT_VIDEO, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))
    
    print("🎬 시연 시작! 종료는 'q'를 누르세요.")

    try:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret: break
            
            results = model.track(frame, device='mps', persist=True, verbose=False)
            
            if results[0].boxes.id is not None:
                # 데이터를 명확하게 일반 정수 리스트/배열로 추출
                boxes = results[0].boxes.xyxy.cpu().numpy()
                cls_ids = results[0].boxes.cls.cpu().numpy().astype(int)
                track_ids = results[0].boxes.id.cpu().numpy().astype(int)
                confs = results[0].boxes.conf.cpu().numpy()
                
                for box, track_id, cls_id, conf in zip(boxes, track_ids, cls_ids, confs):
                    draw_custom_box(frame, box, track_id, cls_id, conf, model.names)
            
            cv2.imshow("Final CCTV Demo", frame)
            out.write(frame)
            
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        cap.release()
        out.release()
        cv2.destroyAllWindows()
        print(f"✅ 저장 완료! 파일 확인: {OUTPUT_VIDEO}")

if __name__ == "__main__":
    run_final_demo()

🎬 시연 시작! 종료는 'q'를 누르세요.
Loading /Users/gyuminkang/Desktop/YOLO/runs/detect/runs/tune_cells/tune_medium/weights/best.mlpackage for CoreML inference...
✅ 저장 완료! 파일 확인: /Users/gyuminkang/Desktop/YOLO/output_final.mp4
